# GSB 5544 — Topic 2.2: Data Wrangling  
*Fill each `____` blank during class; later checks ask you to write complete expressions.*

In [ ]:
import ____ as pd
crime = pd.________("https://raw.githubusercontent.com/gato365/gsb5544_instructor_learn_prep/main/assignments/Data/sc_crime_sample.csv")
crime.______()


In [ ]:
crime.____


---
## 1. *Is this data "tidy"?*

Before wrangling anything, we need a standard for what a **good** table looks like. The book's definition:

> A dataset is **tidy** if
> 1. every **column** is a **variable**,
> 2. every **row** is an **observation**, and
> 3. every **cell** is a **single value**.

Why care? Because `plotnine`, `.groupby`, and nearly every tool we'll use assume it. `aes(x = "crime_cat")` only works if `crime_cat` is a *column* holding *one value per row*.

**Q: Is `crime` tidy?** Look at `crime.______()` above.

- each column (`year`, `crime_cat`, `location_type`, …) is one variable ✔
- each row is one offense record ✔
- each cell holds one value ✔

**Yes.** Now here is the *same information*, arranged the way a report or spreadsheet often arrives:


In [ ]:
wide = pd.read_csv("https://raw.githubusercontent.com/gato365/gsb5544_instructor_learn_prep/main/assignments/Data/sc_crime_by_year_wide.csv")
wide.head()


**Q: Is `wide` tidy?** Count the variables: there are really three — *crime category*, *year*, and *number of offenses*. But `year` is smeared across 30 column **headers**, and the "number of offenses" variable is spread across 30 columns. Rule 1 is broken (column headers are *values*, not variable names), so rule 2 is too (one row holds 30 observations).

**Q: What would I need to do to plot offenses per year for one category?** With `wide` I'd have to hand-pick 30 columns. The fix is to **melt** the wide columns into two long ones — a `year` column and a `n_offenses` column:


In [ ]:
tidy = wide.____(id_vars = "____",        # the column(s) to keep as-is
                 var_name = "____",           # the old column HEADERS become this variable
                 value_name = "____")   # the old cell VALUES become this variable
tidy["year"] = tidy["year"].astype(int)       # headers were text; make year a number
tidy.head()


In [ ]:
tidy.shape        # 22 categories x 30 years = 660 observations, one per row


Now every tool works. (`.melt()` and its opposite, `.pivot()`, get their own chapter next week; today just recognize *untidy* when you see it.)


In [ ]:
from plotnine import *

(ggplot(tidy[tidy["crime_cat"] == "Burglary/Breaking & Entering"], aes(x = "year", y = "n_offenses"))
 + geom_line()
 + labs(title = "Burglaries reported in South Carolina", y = "Offenses")
)


✅ **Check:** below is a tiny table a police chief might send you. Which rule(s) of tidy data does it break, and what would the tidy version's columns be? (Write your answer in the cell — no code needed.)

| agency | 2019_arrests | 2020_arrests |
|---|---|---|
| Columbia | 4,120 | 3,880 |
| Charleston | 2,910 | 2,650 |


**Answer:** the column headers `2019_arrests` / `2020_arrests` are *values* of a year variable (rule 1), so each row holds two observations (rule 2). Tidy columns: `agency`, `year`, `arrests` — four rows.


---
## 2. The Big Five verbs

Almost everything you'll do to a table is one of five operations. If you've seen R's `dplyr`, these are its verbs; pandas spells them differently but they are the same ideas.

| Verb | Question it answers | `dplyr` | pandas |
|---|---|---|---|
| **select** | *Which columns do I need?* | `select()` | `df[["a", "b"]]`, `df.loc[:, ...]` |
| **filter** | *Which rows do I need?* | `filter()` | `df[condition]` |
| **arrange** | *In what order?* | `arrange()` | `df.sort_values()` |
| **mutate** | *What new column do I need?* | `mutate()` | `df["new"] = ...` |
| **summarize** | *What is the one-number (or one-row-per-group) answer?* | `summarize()` | `.mean()`, `.describe()`, `.value_counts()`, `.groupby()` |

We'll meet each one because a question forces us to.


---
## 3. *What is even in this data?*  → `.unique()`, `.nunique()`, `.value_counts()`, `.describe()`

Before asking sharp questions we need the lay of the land. **Q: What kinds of crime are recorded?**


In [ ]:
crime["____"].unique()          # the distinct levels


In [ ]:
crime["____"].nunique()         # how many distinct levels


20 categories in our sample (the full file has 22 — two are so rare they didn't land in 50,000 rows). **Q: Which are common and which are rare?** `.value_counts()` counts rows per level, sorted from most to least:


In [ ]:
crime["____"].value_counts()


In [ ]:
# Q: As a SHARE of all offenses?   normalize=True -> proportions
crime["____"].value_counts(normalize = ____).round(3)


Larceny/theft alone is about a third of all offense records; assault and vandalism are next. Homicide is a few tenths of a percent — rare events need a big sample.

**Q: What does `.describe()` tell me?** It depends on the *type* of the column:


In [ ]:
crime["____"].describe()            # quantitative -> count, mean, sd, min, quartiles, max


In [ ]:
crime["location_type"].describe()   # categorical -> count, #unique, most common level, its frequency


(Is the "mean year" 2006 meaningful? Not really — `year` is an *ordered categorical* here, as with the coffee data last week. `.describe()` computes whatever the dtype allows; **you** decide whether it means anything.)

✅ **Check:** how many distinct `location_type` levels are there, and what share of offenses happen at a `residence/home`?


In [ ]:
print(crime["location_type"].nunique())
crime["location_type"].value_counts(normalize = True).head()


**Answer:** 43 location types; about 45% of offense records are at a residence/home — by far the most common place, ahead of roads and parking lots.


---
## 4. *Which columns do I need?*  → **select**

**Q: For a "what, when, where" report, which columns matter?** Only a few — the IDs and codes are noise for this question. Selecting columns keeps the screen readable and makes later steps faster.


In [ ]:
report = crime[["____", "____", "____", "____", "____"]]
report.head()


(Same `[[ ]]` from last week; `crime.loc[:, ["incident_date", "crime_cat"]]` is equivalent. A *list* of names → DataFrame; a single name → Series.)


---
## 5. *Which rows do I need?*  → **filter**

**Q: How many shoplifting offenses were recorded in 2019?**

A condition on a column gives `True`/`False` per row; putting it inside `[ ]` keeps the `True` rows. Two conditions combine with `&` (and) / `|` (or), each in parentheses.


In [ ]:
shop_2019 = crime[(crime["crime"] == "Shoplifting") & (crime["year"] == 2019)]
shop_2019.shape


In [ ]:
# Q: Which offenses involved ANY kind of firearm?   .isin() = "is one of these"
guns = ["handgun", "rifle", "shotgun", "firearm (type not stated)", "other firearm"]
crime[crime["weapon_force_1"].isin(guns)]["crime"].value_counts().head()


In [ ]:
# Q: How many offense records have NO weapon information at all?   .isna() -> True where missing
crime["weapon_force_1"].isna().sum()


Most rows have no weapon recorded — NIBRS only collects weapons for certain offense types (assault, robbery, homicide, …). **Missing here means "not applicable", not "unknown".** Always check `.isna()` before you compute with a column.

*Fill in the blanks.*

✅ **Check:** how many offenses in the sample were **attempted but not completed** (`completed == False`) **and** ended in an arrest?


In [ ]:
crime[(crime["completed"] == False) & (crime["arrest_made"] == True)].shape[0]


---
## 6. *In what order?*  → **arrange**

**Q: What are the most recent offenses in the sample?**

`.sort_values()` sorts by one or more columns. `ascending = False` reverses.


In [ ]:
crime.sort_values("incident_date", ascending = False).head()


In [ ]:
# Q: Within each year, which categories come first alphabetically?   a list sorts by several columns, in order
crime.sort_values(["year", "crime_cat"]).head()


**Nothing changed.** Sorting (and selecting, and filtering) *returns a new table*; `crime` itself is untouched:


In [ ]:
crime.head(2)          # still the original order


To keep a result, **assign it**: `crime = crime.sort_values(...)`. The book warns against `inplace=True` — an overwrite is a big decision and should be visible on the left of an `=`.

✅ **Check:** among `Homicide Offenses`, which agency (`ori`) has the most records? (Hint: filter, then `.value_counts()`; or filter then sort.)


In [ ]:
crime[crime["crime_cat"] == "Homicide Offenses"]["ori"].value_counts().head(3)


---
## 7. *What new column do I need?*  → **mutate**

New columns are made by assigning to a name that doesn't exist yet: `crime["new"] = <expression>`. The four moves from the book:

**Q: Is an offense violent, property, or something else?** The 22 categories are too fine. **Recode** with `.map()` (last week's tool) — anything not listed becomes missing, then `.fillna()` catches the rest:


In [ ]:
violent  = ["Assault Offenses", "Homicide Offenses", "Robbery", "Kidnapping/Abduction", "Human Trafficking"]
property = ["Larceny/Theft Offenses", "Burglary/Breaking & Entering", "Motor Vehicle Theft", "Arson",
            "Destruction/Damage/Vandalism of Property", "Stolen Property Offenses"]

crime["crime_group"] = (crime["crime_cat"]
                        .map({**{c: "violent" for c in violent}, **{c: "property" for c in property}})
                        .fillna("other")
                        .astype("category"))
crime["crime_group"].value_counts()


**Q: Which decade?** `year` is quantitative-looking, so arithmetic works: integer-divide by 10 and multiply back.


In [ ]:
crime["decade"] = (crime["year"] // 10 * 10).astype(str) + "s"
crime["decade"].value_counts().sort_index()


**Q: Was a firearm involved?** A **boolean** column from a condition. `1 * (...)` turns `True/False` into `1/0` — handy for later modeling.


In [ ]:
crime["firearm"] = 1 * crime["weapon_force_1"].isin(guns)
crime["firearm"].value_counts()


**Q: What day of the week?** `incident_date` is text. Convert it to a real date (`pd.to_datetime`, as with the coffee data), then pull out the weekday.


In [ ]:
crime["incident_date"] = pd.to_datetime(crime["incident_date"])
crime["weekday"] = crime["incident_date"].dt.day_name()
crime["weekday"].value_counts()


`.value_counts()` sorts by count, so the weekdays are out of order. Same fix as last week — declare an **ordered** categorical:


In [ ]:
days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
crime["weekday"] = pd.Categorical(crime["weekday"], categories = days, ordered = True)
crime["weekday"].value_counts().sort_index()


✅ **Check:** make a column `season` from `month` (`Dec–Feb` = "winter", `Mar–May` = "spring", `Jun–Aug` = "summer", `Sep–Nov` = "fall") and show the counts. Use `.map()` or `pd.cut()` — which is more natural here, and why?


In [ ]:
season_map = {12: "winter", 1: "winter", 2: "winter", 3: "spring", 4: "spring", 5: "spring",
              6: "summer", 7: "summer", 8: "summer", 9: "fall", 10: "fall", 11: "fall"}
crime["season"] = crime["month"].map(season_map).astype("category")
crime["season"].value_counts()


**Answer:** `.map()` — `pd.cut()` bins a *range* of numbers, but December (12) and January (1) belong to the same season, which no single set of cut points can express. Summer is the busiest season.


---
## 8. *What is the one-number answer?*  → **summarize**

**Q: What share of offenses end in an arrest?**

`arrest_made` is `True`/`False`. Python treats `True` as 1, so the **mean of a boolean is a proportion**.


In [ ]:
crime["arrest_made"].mean()


About one in four. But that single number hides the interesting part.

**Q: Which kinds of crime are most likely to end in an arrest?** Split the rows into groups, then summarize *each group*: `.groupby("group")["column"].statistic()`.


In [ ]:
crime.groupby("crime_cat")["arrest_made"].mean().sort_values(ascending = False).round(3)


Prostitution, drug, and weapon offenses are usually "cleared" — the offense *is* the arrest. Larceny, burglary, and vandalism are rarely solved. **Reported arrest rate says as much about how a crime is discovered as about policing.**


In [ ]:
# Q: Several summaries at once?   .agg() with a dictionary: column -> statistic(s)
crime.groupby("crime_group").agg(n = ("crime_cat", "size"),
                                 arrest_rate = ("arrest_made", "mean"),
                                 firearm_rate = ("firearm", "mean")).round(3)


✅ **Check:** has the arrest rate changed over time? Compute it by `decade`, then by `year`, and plot the yearly series with `geom_line`.


In [ ]:
print(crime.groupby("decade")["arrest_made"].mean().round(3))

by_year = crime.groupby("year")["arrest_made"].mean().reset_index()
(ggplot(by_year, aes(x = "year", y = "arrest_made"))
 + geom_line()
 + geom_point()
 + labs(y = "Share of offenses with an arrest", title = "South Carolina, 1991-2020 (sample)")
)


**Answer:** the arrest share *rises*, from about 21% in the 1990s to about 27% in the 2010s. Before concluding "policing got better", remember Section 8: drug and weapon offenses are arrests almost by definition, and PA 5 below shows they became a bigger share of what gets recorded.


---
## 9. *Do two categorical variables go together?*  → `pd.crosstab()`

**Q: Are attempted offenses less likely to end in an arrest than completed ones?**

`.value_counts()` describes one categorical variable. For **two**, we need a **cross-tabulation** (a "crosstab" or contingency table): rows = levels of one, columns = levels of the other, cells = counts.


In [ ]:
pd.crosstab(crime["completed"], crime["arrest_made"])


Counts alone don't answer the question — there are far more completed offenses. We need **conditional proportions**: *within each row*, what share had an arrest? `normalize = "index"` divides each row by its total.

| `normalize=` | Each cell is … |
|---|---|
| `False` (default) | a count |
| `True` | a share of the **grand total** |
| `"index"` | a share of its **row** |
| `"columns"` | a share of its **column** |


In [ ]:
pd.crosstab(crime["completed"], crime["arrest_made"], normalize = "index").round(3)


About 24% of completed offenses lead to an arrest vs. about 21% of attempted ones — a modest gap, in the direction we guessed.


In [ ]:
# Q: Where do violent vs. property vs. other offenses happen?   margins=True adds row/column totals
pd.crosstab(crime["crime_group"], crime["location_type"], margins = True).iloc[:, :6]   # first 6 locations


✅ **Check:** does the weekday pattern differ by `crime_group`? Build the crosstab of `crime_group` (rows) by `weekday` (columns) as **row proportions**. Which group is most concentrated on weekends?


In [ ]:
pd.crosstab(crime["crime_group"], crime["weekday"], normalize = "index").round(3)


**Answer:** violent offenses tilt toward Saturday/Sunday (~16% each vs ~13% midweek); property offenses are flat across the week, with a slight Friday bump.


## 10. Bridge to PA 2.2: grouped transformations and wide data

The practice activity combines the verbs above. Use this pattern: **identify the observational unit → filter/select → mutate → group → summarize → arrange**.

### A. Several grouping variables

Pass a list to `groupby` when each answer needs more than one label. The mean of a 0/1 indicator is a proportion.

In [ ]:
# One survival proportion for every gender/type combination
titanic_example = pd.DataFrame({
    "gender": ["female", "female", "male", "male"],
    "type": ["passenger", "crew", "passenger", "crew"],
    "survived": [1, 1, 0, 1]
})
titanic_example.groupby(["____", "____"])["____"].mean()

### B. Weighted means and `transform`

A weighted mean is `(value * weight).sum() / weight.sum()`. `transform` returns one value per original row, so it is useful when each row needs its group's total.

In [ ]:
college_example = pd.DataFrame({
    "state": ["CA", "CA", "OR"],
    "earnings": [70000, 50000, 60000],
    "enrollment": [20000, 5000, 10000]
})
college_example["state_enrollment"] = (
    college_example.groupby("____")["____"].transform("____")
)
college_example["state_weight"] = (
    college_example["____"] / college_example["____"]
)
college_example["weighted_earnings"] = (
    college_example["earnings"] * college_example["state_weight"]
)
college_example.groupby("state")["weighted_earnings"].sum()

### C. Missing values, replacing, and vectorized standardization

Use `.dropna(subset=[...])` to filter missing rows and `.replace(0, pd.NA)` when a sentinel value means missing. DataFrame arithmetic works column-by-column, without loops.

In [ ]:
scores_example = pd.DataFrame({"floor": [12.0, 13.0, 0], "rings": [11.0, 14.0, 15.0]})
scores_example = scores_example.replace(____, pd.____)
standardized = (scores_example - scores_example.____()) / scores_example.____()
standardized["total_z"] = standardized.sum(axis=____)
standardized.sort_values("total_z", ascending=False)

### D. Reshape: index, wide, and long

`.set_index([...])` identifies rows without deleting the score columns. `.melt()` stacks several score columns into an event column and a score column. `.pivot()` spreads values back into columns.

In [ ]:
gym_example = pd.DataFrame({
    "year": [2020, 2020], "gymnast": ["A", "B"],
    "floor": [14.1, 13.8], "rings": [13.9, 14.2]
})
gym_indexed = gym_example.set_index(["____", "____"])[["floor", "rings"]]
gym_long = gym_indexed.reset_index().melt(
    id_vars=["____", "____"], var_name="____", value_name="____"
)
gym_long

### PA 2.2 readiness check

Before starting, make sure you can explain when to use `value_counts(normalize=True)`, the mean of an indicator, `groupby([...])`, `describe()`, `dropna`, `transform`, DataFrame arithmetic, `sum(axis=1)`, `set_index`, and `melt`.

---
## 11. Practice Activities — you pick the verb

For each question, first say **which verb(s)** (select / filter / arrange / mutate / summarize / crosstab) you need, then write the code. Solutions will be posted after class.

**PA 1.** What are the five most common *specific* offenses (`crime`) among **property** crimes?


In [ ]:
# filter -> summarize
crime[crime["crime_group"] == "property"]["crime"].value_counts().head(5)


**PA 2.** Where does shoplifting happen? Show the top 5 `location_type` values for `Shoplifting` as *proportions*.


In [ ]:
# filter -> summarize (normalize)
crime[crime["crime"] == "Shoplifting"]["location_type"].value_counts(normalize = True).head(5).round(3)


**PA 3.** Create a boolean column `hate_crime` that is `True` when `bias_motivation` is recorded and is **not** `"no bias motivation"` or `"unknown bias motivation"`. How many hate-crime records are in the sample, and which `crime_cat` do they fall into most often?


In [ ]:
# mutate -> summarize
not_bias = ["no bias motivation", "unknown bias motivation"]
crime["hate_crime"] = crime["bias_motivation"].notna() & ~crime["bias_motivation"].isin(not_bias)
print(crime["hate_crime"].sum())
crime[crime["hate_crime"]]["crime_cat"].value_counts().head(3)


**PA 4.** Is the firearm rate different for violent offenses at a `residence/home` versus on a `highway/road/alley`? (filter → groupby → mean)


In [ ]:
# filter -> summarize by group
v = crime[(crime["crime_group"] == "violent") & (crime["location_type"].isin(["residence/home", "highway/road/alley"]))]
v.groupby("location_type")["firearm"].mean().round(3)


**PA 5.** Build a crosstab of `decade` by `crime_group` as **row proportions**. Has the *mix* of reported crime shifted across decades? Then draw it: `geom_bar(position = "fill")` with `x = "decade"` and `fill = "crime_group"`.


In [ ]:
print(pd.crosstab(crime["decade"], crime["crime_group"], normalize = "index").round(3))

(ggplot(crime, aes(x = "decade", fill = "crime_group"))
 + geom_bar(position = "fill")
 + labs(y = "Share of offense records", title = "Mix of reported crime by decade")
)


**Answer:** property crime's share falls across the decades while "other" (mostly drug offenses) rises — a change in what police *record* as much as in what happens.

**PA 6.** The table below is not tidy. Type it into a DataFrame with `pd.DataFrame({...})`, melt it into tidy form, and then answer: which agency had the larger *increase* in offenses from 2018 to 2020?

| agency | offenses_2018 | offenses_2019 | offenses_2020 |
|---|---|---|---|
| Greenville | 5100 | 5300 | 4900 |
| Myrtle Beach | 3900 | 4300 | 4600 |


In [ ]:
agencies = pd.DataFrame({"agency": ["Greenville", "Myrtle Beach"],
                         "offenses_2018": [5100, 3900],
                         "offenses_2019": [5300, 4300],
                         "offenses_2020": [4900, 4600]})

long = agencies.melt(id_vars = "agency", var_name = "____", value_name = "offenses")
long["year"] = long["year"].str.replace("offenses_", "").astype(int)

change = long[long["year"] == 2020].set_index("agency")["offenses"] - long[long["year"] == 2018].set_index("agency")["offenses"]
change.sort_values(ascending = False)


**Answer:** Myrtle Beach rose by 700; Greenville fell by 200.

**PA 7 (write, don't code).** Section 8 showed the arrest share rising from the 1990s to the 2010s. A news story says "South Carolina police got 30% better at solving crimes." Using what you saw in Sections 8–9 and PA 5, give two reasons the *data* might show that rise even if policing did not change.


**Answer:** (1) the *mix* changed — drug and weapon offenses (arrest rates of 70–80%, because the offense is discovered *by* the arrest) grew from ~10% to ~25% of records, which raises the overall rate even if every category's own rate stayed flat; (2) this is *reported and recorded* crime — which agencies report, and what they record, changed over 30 years. A fair comparison holds the crime mix fixed (compare rates *within* a category).


---
## Summary

| Question | Verb | Tool |
|---|---|---|
| Is this table tidy? | — | every column a variable, every row an observation, every cell one value; `.melt()` to fix wide data |
| What's in this column? | summarize | `.unique()`, `.nunique()`, `.value_counts(normalize=)`, `.describe()` |
| Which columns? | **select** | `df[["a", "b"]]`, `df.loc[:, [...]]` |
| Which rows? | **filter** | `df[cond]`, `&`, `|`, `.isin()`, `.isna()` |
| In what order? | **arrange** | `.sort_values(cols, ascending=)` — assign to keep |
| What new column? | **mutate** | `df["new"] = ...`, `.map()`, arithmetic, `1 * (cond)`, `pd.to_datetime`, `.dt.day_name()` |
| One-number answer, overall or per group? | **summarize** | `.mean()` (of a boolean = proportion), `.groupby("g")["x"].mean()`, `.agg()` |
| Do two categoricals go together? | crosstab | `pd.crosstab(a, b, normalize="index", margins=True)` |
